In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
df.isnull().sum()

In [ ]:
(df["TotalCharges"]==" ").sum()

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.drop('customerID', axis=1, inplace=True, errors='ignore')

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
for col in df.columns:
    print("="*50)
    print(col)
    print(df[col].unique())
    print("Number of unique values:", df[col].nunique())

In [ ]:
internet_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in internet_cols:
    df[col] = df[col].replace("No internet service", "No")

In [ ]:
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

In [ ]:
for col in internet_cols + ["MultipleLines"]:
    print("="*40)
    print(col)
    print(df[col].unique())

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = le.fit_transform(df[col])

In [ ]:
df.info()

In [ ]:
plt.figure(figsize=(18,10))

sns.heatmap(
    df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")
plt.show()

In [ ]:
corr = df.corr()

corr["Churn"].sort_values(ascending=False)

In [ ]:
corr = df.corr()

target_corr = corr["Churn"].abs()

threshold = 0.05

selected_features = target_corr[target_corr >= threshold].index

print(selected_features)

In [ ]:
df = df[selected_features]

In [ ]:
X = df.drop("Churn", axis=1)

y = df["Churn"]

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
lr = LogisticRegression(
    C=1,
    max_iter=1000,
    random_state=42
)

lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))

In [ ]:
print(classification_report(y_test, y_pred_lr))

In [ ]:
plt.figure(figsize=(5,4))

sns.heatmap(confusion_matrix(y_test, y_pred_lr),
            annot=True,
            fmt="d",
            cmap="Blues")

plt.title("Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
dt = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    min_samples_split=5,
    random_state=42
)

dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_dt))

In [ ]:
plt.figure(figsize=(5,4))

sns.heatmap(confusion_matrix(y_test, y_pred_dt),
            annot=True,
            fmt="d",
            cmap="Blues")

plt.title("Decision Tree")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
print(classification_report(y_test, y_pred_dt))

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))

In [ ]:
print(classification_report(y_test, y_pred_rf))

In [ ]:
plt.figure(figsize=(5,4))

sns.heatmap(confusion_matrix(y_test, y_pred_rf),
            annot=True,
            fmt="d",
            cmap="Blues")

plt.title("Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf)
    ]
})

results.sort_values(by="Accuracy", ascending=False)

In [ ]:
df.shape